# PySpark practice 7: aggregations

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-practice-07-aggregations.ipynb)

16 exercises on grouping, aggregate functions, conditional and approximate aggregates, pivots and subtotals. Each exercise names the [reference entries](pyspark-reference-07-aggregations.ipynb) it practises, shows its inputs and the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local session, the shared `customers`, `orders`, `order_items`, `products` and `events` tables (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry), and the helpers from [PySpark reference 7](pyspark-reference-07-aggregations.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import Window
from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

products = spark.createDataFrame([
    ("SKU-4001", "Kitchen", D("12.50")), ("SKU-4002", "Kitchen", D("34.50")),
    ("SKU-4003", "Garden", D("149.00")), ("SKU-4004", "Garden", D("55.50")),
    ("SKU-4005", "Pantry", D("55.25")), ("SKU-4006", "Outdoor", D("189.00")),
    ("SKU-4007", "Kitchen", D("95.00")), ("SKU-4008", "Outdoor", D("121.00")),
    ("SKU-4009", "Pantry", D("25.00")), ("SKU-4010", "Garden", D("89.90")),
    ("SKU-4011", "Pantry", D("59.00")), ("SKU-4012", "Kitchen", D("37.00")),
    ("SKU-4013", "Outdoor", D("210.00")),  # SKU-4014 is missing
], "sku string, category string, list_price decimal(8,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)), (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)), (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)), (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)), (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)), (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)), (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

import contextlib
import io
import logging
import re


def plan_lines(df, mode="simple"):
    """The lines df.explain(mode) prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue().splitlines()


def final_plan(df):
    """Run df, then return the plan Spark actually used (AQE's final plan when there is one)."""
    df.collect()
    text = "\n".join(plan_lines(df))
    if "== Final Plan ==" in text:
        text = text.split("== Final Plan ==")[1].split("== Initial Plan ==")[0]
    return text


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        print(type(error).__name__ + ":", str(error).split(".")[0][:100])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

ads = spark.createDataFrame(
    [("north", 10, 120), ("north", 20, 190), ("north", 30, 310), ("north", 40, 390),
     ("south", 10, 200), ("south", 20, 180), ("south", 30, 220), ("south", None, 210)],
    "region string, spend int, sales int",
)

## Exercises

5 easy, 8 medium, 3 hard. Write your answer in the empty cell under each exercise as a DataFrame called `result`, run it, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Revenue and order count per status](#exercise-1-revenue-and-order-count-per-status) | Easy | [`DataFrame.groupBy(*cols)` and `GroupedData`](pyspark-reference-07-aggregations.ipynb#dataframegroupbycols-and-groupeddata) |
| 2 | [Daily KPIs in one pass](#exercise-2-daily-kpis-in-one-pass) | Medium | [`sum(when(...))`, `count_if` and SQL `FILTER`](pyspark-reference-07-aggregations.ipynb#sumwhen-count_if-and-sql-filter) |
| 3 | [Customers with at least two orders and 300 of revenue](#exercise-3-customers-with-at-least-two-orders-and-300-of-revenue) | Easy | [Filtering groups (`HAVING`)](pyspark-reference-07-aggregations.ipynb#filtering-groups-having) |
| 4 | [Every region including unknown and empty ones](#exercise-4-every-region-including-unknown-and-empty-ones) | Medium | [Keeping groups that have no rows](pyspark-reference-07-aggregations.ipynb#keeping-groups-that-have-no-rows) |
| 5 | [Latest status and biggest order per customer](#exercise-5-latest-status-and-biggest-order-per-customer) | Medium | [`max_by`, `min_by`](pyspark-reference-07-aggregations.ipynb#max_by-min_by) |
| 6 | [Status history as a sorted list](#exercise-6-status-history-as-a-sorted-list) | Easy | [`collect_list` / `array_agg`, `collect_set`](pyspark-reference-07-aggregations.ipynb#collect_list--array_agg-collect_set), [`listagg` / `string_agg`](pyspark-reference-07-aggregations.ipynb#listagg--string_agg) |
| 7 | [Median vs mean per category](#exercise-7-median-vs-mean-per-category) | Medium | [`median`, `percentile`](pyspark-reference-07-aggregations.ipynb#median-percentile) |
| 8 | [Pivot revenue by status](#exercise-8-pivot-revenue-by-status) | Easy | [`GroupedData.pivot(pivot_col, values=None)`](pyspark-reference-07-aggregations.ipynb#groupeddatapivotpivot_col-valuesnone) |
| 9 | [Unpivot monthly columns into rows](#exercise-9-unpivot-monthly-columns-into-rows) | Medium | [`DataFrame.unpivot(ids, values, variableColumnName, valueColumnName)` / `melt`](pyspark-reference-07-aggregations.ipynb#dataframeunpivotids-values-variablecolumnname-valuecolumnname--melt) |
| 10 | [Subtotals with rollup](#exercise-10-subtotals-with-rollup) | Medium | [`rollup`, `cube` and `GROUPING SETS`](pyspark-reference-07-aggregations.ipynb#rollup-cube-and-grouping-sets), [`grouping`, `grouping_id`](pyspark-reference-07-aggregations.ipynb#grouping-grouping_id) |
| 11 | [Funnel conversion per user](#exercise-11-funnel-conversion-per-user) | Hard | [`sum(when(...))`, `count_if` and SQL `FILTER`](pyspark-reference-07-aggregations.ipynb#sumwhen-count_if-and-sql-filter), [`bool_and` / `every`, `bool_or` / `some`](pyspark-reference-07-aggregations.ipynb#bool_and--every-bool_or--some) |
| 12 | [Distinct users per week from daily sketches](#exercise-12-distinct-users-per-week-from-daily-sketches) | Hard | [HyperLogLog sketches: `hll_sketch_agg`, `hll_union_agg`, `hll_sketch_estimate`](pyspark-reference-07-aggregations.ipynb#hyperloglog-sketches-hll_sketch_agg-hll_union_agg-hll_sketch_estimate) |
| 13 | [Top category per region by revenue](#exercise-13-top-category-per-region-by-revenue) | Hard | [`max_by`, `min_by`](pyspark-reference-07-aggregations.ipynb#max_by-min_by) |
| 14 | [Profile a table's columns](#exercise-14-profile-a-tables-columns) | Medium | [`count`, `count_distinct`](pyspark-reference-07-aggregations.ipynb#count-count_distinct), [`approx_count_distinct`](pyspark-reference-07-aggregations.ipynb#approx_count_distinct) |
| 15 | [Weighted average price per category](#exercise-15-weighted-average-price-per-category) | Medium | [`sum`, `avg` / `mean`, `min`, `max`](pyspark-reference-07-aggregations.ipynb#sum-avg--mean-min-max) |
| 16 | [GROUP BY ALL with FILTER in SQL](#exercise-16-group-by-all-with-filter-in-sql) | Easy | [`GROUP BY`, `GROUP BY ALL`, `HAVING` and `FILTER`](pyspark-reference-07-aggregations.ipynb#group-by-group-by-all-having-and-filter) |

## Exercise 1: Revenue and order count per status

**Difficulty:** Easy · about 5 min · **Practises:** [`DataFrame.groupBy(*cols)` and `GroupedData`](pyspark-reference-07-aggregations.ipynb#dataframegroupbycols-and-groupeddata)

*The first aggregation anyone asks for.*

Return `status`, `orders` and `revenue` per status, ordered by `revenue` descending.

In [ ]:
orders.select("order_id", "status", "amount").show()

**Expected output**

| status | orders | revenue |
| --- | --- | --- |
| shipped | 4 | 1509.90 |
| paid | 4 | 764.00 |
| placed | 2 | 265.25 |
| returned | 2 | 119.90 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.groupBy("status").agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue")).orderBy(F.desc("revenue"))
result.show()

## Exercise 2: Daily KPIs in one pass

**Difficulty:** Medium · about 10 min · **Practises:** [`sum(when(...))`, `count_if` and SQL `FILTER`](pyspark-reference-07-aggregations.ipynb#sumwhen-count_if-and-sql-filter)

*KPI tables compute many filtered metrics per day; one scan, not one query per metric.*

Per UTC order date, return `day`, `orders`, `paid_revenue` (0 when none), `returns` and `open_orders` (status `placed`), ordered by `day`.

In [ ]:
orders.select("order_id", "status", "order_ts", "amount").show()

**Expected output**

| day | orders | paid_revenue | returns | open_orders |
| --- | --- | --- | --- | --- |
| 2026-01-05 | 3 | 120.00 | 1 | 0 |
| 2026-01-06 | 1 | 260.00 | 0 | 0 |
| 2026-01-07 | 2 | 0.00 | 0 | 1 |
| 2026-01-09 | 1 | 310.00 | 0 | 0 |
| 2026-01-10 | 1 | 0.00 | 1 | 0 |
| 2026-01-12 | 2 | 74.00 | 0 | 0 |
| 2026-01-13 | 1 | 0.00 | 0 | 1 |
| 2026-01-14 | 1 | 0.00 | 0 | 0 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.groupBy(F.to_date("order_ts").alias("day")).agg(
    F.count("*").alias("orders"),
    F.coalesce(F.sum(F.when(F.col("status") == "paid", F.col("amount"))), F.lit(0)).alias("paid_revenue"),
    F.count_if(F.col("status") == "returned").alias("returns"),
    F.count_if(F.col("status") == "placed").alias("open_orders"),
).orderBy("day")
result.show()

## Exercise 3: Customers with at least two orders and 300 of revenue

**Difficulty:** Easy · about 5 min · **Practises:** [Filtering groups (`HAVING`)](pyspark-reference-07-aggregations.ipynb#filtering-groups-having)

*HAVING-style filters are a staple of interview SQL.*

Return `customer_id`, `orders` and `revenue` for customers with 2 or more orders and revenue of at least 300, ordered by `customer_id`.

In [ ]:
orders.select("order_id", "customer_id", "amount").show()

**Expected output**

| customer_id | orders | revenue |
| --- | --- | --- |
| 2 | 2 | 315.25 |
| 3 | 2 | 459.00 |
| 4 | 2 | 1269.90 |
| 6 | 2 | 306.40 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = (orders.groupBy("customer_id").agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue"))
          .where((F.col("orders") >= 2) & (F.col("revenue") >= 300)).orderBy("customer_id"))
result.show()

## Exercise 4: Every region including unknown and empty ones

**Difficulty:** Medium · about 10 min · **Practises:** [Keeping groups that have no rows](pyspark-reference-07-aggregations.ipynb#keeping-groups-that-have-no-rows)

*A region with no sales must still appear with 0.*

Return every region in `regions` plus `unknown` (customers without a region) with `revenue` (0 when none), ordered by `region`.

In [ ]:
regions = spark.createDataFrame([("North",), ("South",), ("East",), ("West",), ("Central",), ("unknown",)], "region string")
regions.show()

**Expected output**

| region | revenue |
| --- | --- |
| Central | 0.00 |
| East | 0.00 |
| North | 549.75 |
| South | 459.00 |
| West | 380.40 |
| unknown | 1269.90 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
revenue = (orders.join(customers, "customer_id", "left")
           .groupBy(F.coalesce("region", F.lit("unknown")).alias("region")).agg(F.sum("amount").alias("revenue")))
result = regions.join(revenue, "region", "left").select("region", F.coalesce("revenue", F.lit(0)).alias("revenue")).orderBy("region")
result.show()

## Exercise 5: Latest status and biggest order per customer

**Difficulty:** Medium · about 10 min · **Practises:** [`max_by`, `min_by`](pyspark-reference-07-aggregations.ipynb#max_by-min_by)

*Picking a value from a specific row without a window or a join back.*

Return `customer_id`, `latest_status` (status of the most recent order), `biggest_order_id` and `first_order_date`, ordered by `customer_id`.

In [ ]:
orders.select("order_id", "customer_id", "status", "order_ts", "amount").show()

**Expected output**

| customer_id | latest_status | biggest_order_id | first_order_date |
| --- | --- | --- | --- |
| 1 | returned | 1 | 2026-01-05 |
| 2 | placed | 4 | 2026-01-06 |
| 3 | paid | 7 | 2026-01-07 |
| 4 | shipped | 9 | 2026-01-10 |
| 5 | paid | 10 | 2026-01-12 |
| 6 | shipped | 11 | 2026-01-13 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.groupBy("customer_id").agg(
    F.max_by("status", "order_ts").alias("latest_status"),
    F.max_by("order_id", "amount").alias("biggest_order_id"),
    F.to_date(F.min("order_ts")).alias("first_order_date"),
).orderBy("customer_id")
result.show()

## Exercise 6: Status history as a sorted list

**Difficulty:** Easy · about 5 min · **Practises:** [`collect_list` / `array_agg`, `collect_set`](pyspark-reference-07-aggregations.ipynb#collect_list--array_agg-collect_set), [`listagg` / `string_agg`](pyspark-reference-07-aggregations.ipynb#listagg--string_agg)

*Support tools show each customer's statuses as one string.*

Return `customer_id`, `statuses` (distinct statuses sorted and joined with `, `) and `history` (all statuses in time order joined with ` > `), ordered by `customer_id`.

In [ ]:
orders.select("customer_id", "status", "order_ts").orderBy("customer_id", "order_ts").show()

**Expected output**

| customer_id | statuses | history |
| --- | --- | --- |
| 1 | paid, returned, shipped | paid > shipped > returned |
| 2 | paid, placed | paid > placed |
| 3 | paid, shipped | shipped > paid |
| 4 | returned, shipped | returned > shipped |
| 5 | paid | paid |
| 6 | placed, shipped | placed > shipped |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
in_time_order = F.array_sort(F.collect_list(F.struct("order_ts", "status")))
result = orders.groupBy("customer_id").agg(
    F.array_join(F.sort_array(F.collect_set("status")), ", ").alias("statuses"),
    F.array_join(F.transform(in_time_order, lambda s: s["status"]), " > ").alias("history"),
).orderBy("customer_id")
result.show(truncate=False)

## Exercise 7: Median vs mean per category

**Difficulty:** Medium · about 10 min · **Practises:** [`median`, `percentile`](pyspark-reference-07-aggregations.ipynb#median-percentile)

*One large order distorts an average; the median shows the typical value.*

Join `order_items` to `products` and, per `category`, return `lines`, `avg_line` and `median_line` (line value = qty × unit_price, both rounded to 2), ordered by `category`.

In [ ]:
order_items.join(products, "sku").select("order_id", "sku", "category", "qty", "unit_price").show(20)

**Expected output**

| category | lines | avg_line | median_line |
| --- | --- | --- | --- |
| Garden | 4 | 124.73 | 130.0 |
| Kitchen | 5 | 51.70 | 34.5 |
| Outdoor | 3 | 173.33 | 189.0 |
| Pantry | 4 | 335.06 | 55.13 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
line = F.col("qty") * F.col("unit_price")
result = order_items.join(products, "sku").groupBy("category").agg(
    F.count("*").alias("lines"), F.round(F.avg(line), 2).alias("avg_line"), F.round(F.median(line), 2).alias("median_line")
).orderBy("category")
result.show()

## Exercise 8: Pivot revenue by status

**Difficulty:** Easy · about 5 min · **Practises:** [`GroupedData.pivot(pivot_col, values=None)`](pyspark-reference-07-aggregations.ipynb#groupeddatapivotpivot_col-valuesnone)

*Spreadsheet-style reports have one column per status.*

Return one row per customer with columns `customer_id`, `paid`, `shipped`, `returned` (revenue, 0 when none), ordered by `customer_id`.

In [ ]:
orders.select("customer_id", "status", "amount").show()

**Expected output**

| customer_id | paid | shipped | returned |
| --- | --- | --- | --- |
| 1 | 120.00 | 84.50 | 30.00 |
| 2 | 260.00 | 0.00 | 0.00 |
| 3 | 310.00 | 149.00 | 0.00 |
| 4 | 0.00 | 1180.00 | 89.90 |
| 5 | 74.00 | 0.00 | 0.00 |
| 6 | 0.00 | 96.40 | 0.00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = orders.groupBy("customer_id").pivot("status", ["paid", "shipped", "returned"]).agg(F.sum("amount")).fillna(0).orderBy("customer_id")
result.show()

## Exercise 9: Unpivot monthly columns into rows

**Difficulty:** Medium · about 10 min · **Practises:** [`DataFrame.unpivot(ids, values, variableColumnName, valueColumnName)` / `melt`](pyspark-reference-07-aggregations.ipynb#dataframeunpivotids-values-variablecolumnname-valuecolumnname--melt)

*Vendor spreadsheets put each month in its own column; the warehouse wants one row per month.*

Unpivot `budget` into `store`, `month`, `amount`, dropping null amounts, ordered by `store`, `month`.

In [ ]:
budget = spark.createDataFrame([("S1", 100, 120, None), ("S2", 80, None, 95)], "store string, jan int, feb int, mar int")
budget.show()

**Expected output**

| store | month | amount |
| --- | --- | --- |
| S1 | feb | 120 |
| S1 | jan | 100 |
| S2 | jan | 80 |
| S2 | mar | 95 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = budget.unpivot("store", ["jan", "feb", "mar"], "month", "amount").where("amount IS NOT NULL").orderBy("store", "month")
result.show()

## Exercise 10: Subtotals with rollup

**Difficulty:** Medium · about 15 min · **Practises:** [`rollup`, `cube` and `GROUPING SETS`](pyspark-reference-07-aggregations.ipynb#rollup-cube-and-grouping-sets), [`grouping`, `grouping_id`](pyspark-reference-07-aggregations.ipynb#grouping-grouping_id)

*Finance reports need detail, subtotals and a grand total from one query, with real null keys kept apart.*

Roll revenue up by `region` then `status` (customers' regions; customer 4 has none). Return `region`, `status`, `revenue` and `level` (`detail`, `region_total` or `grand_total` from `grouping_id`). Order detail rows first, then region totals, then the grand total; within a level by `region` (nulls last) and `status`.

In [ ]:
orders.join(customers.select("customer_id", "region"), "customer_id", "left").select("order_id", "region", "status", "amount").show()

**Expected output**

| region | status | revenue | level |
| --- | --- | --- | --- |
| North | paid | 380.00 | detail |
| North | placed | 55.25 | detail |
| North | returned | 30.00 | detail |
| North | shipped | 84.50 | detail |
| South | paid | 310.00 | detail |
| South | shipped | 149.00 | detail |
| West | paid | 74.00 | detail |
| West | placed | 210.00 | detail |
| West | shipped | 96.40 | detail |
| null | returned | 89.90 | detail |
| null | shipped | 1180.00 | detail |
| North | null | 549.75 | region_total |
| South | null | 459.00 | region_total |
| West | null | 380.40 | region_total |
| null | null | 1269.90 | region_total |
| null | null | 2659.05 | grand_total |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
levels = F.when(F.grouping_id() == 0, "detail").when(F.grouping_id() == 1, "region_total").otherwise("grand_total")
result = (orders.join(customers.select("customer_id", "region"), "customer_id", "left")
          .rollup("region", "status").agg(F.sum("amount").alias("revenue"), levels.alias("level"), F.grouping_id().alias("gid"))
          .orderBy("gid", F.asc_nulls_last("region"), "status").drop("gid"))
result.show(30)

## Exercise 11: Funnel conversion per user

**Difficulty:** Hard · about 15 min · **Practises:** [`sum(when(...))`, `count_if` and SQL `FILTER`](pyspark-reference-07-aggregations.ipynb#sumwhen-count_if-and-sql-filter), [`bool_and` / `every`, `bool_or` / `some`](pyspark-reference-07-aggregations.ipynb#bool_and--every-bool_or--some)

*Product analytics: how many users reached each funnel step.*

After removing duplicate events, compute per user whether they viewed, added to cart and purchased; then return one row with `users`, `viewed`, `carted`, `purchased` and `view_to_purchase` (purchasers who also viewed, divided by viewers, rounded to 2).

In [ ]:
events.orderBy("user_id", "ts").show()

**Expected output**

| users | viewed | carted | purchased | view_to_purchase |
| --- | --- | --- | --- | --- |
| 5 | 4 | 2 | 4 | 0.75 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
per_user = events.dropDuplicates().groupBy("user_id").agg(
    F.bool_or(F.col("event") == "view").alias("viewed"),
    F.bool_or(F.col("event") == "add_to_cart").alias("carted"),
    F.bool_or(F.col("event") == "purchase").alias("purchased"),
)
result = per_user.agg(
    F.count("*").alias("users"),
    F.count_if("viewed").alias("viewed"),
    F.count_if("carted").alias("carted"),
    F.count_if("purchased").alias("purchased"),
    F.round(F.count_if(F.col("viewed") & F.col("purchased")) / F.count_if("viewed"), 2).alias("view_to_purchase"),
)
result.show()

## Exercise 12: Distinct users per week from daily sketches

**Difficulty:** Hard · about 15 min · **Practises:** [HyperLogLog sketches: `hll_sketch_agg`, `hll_union_agg`, `hll_sketch_estimate`](pyspark-reference-07-aggregations.ipynb#hyperloglog-sketches-hll_sketch_agg-hll_union_agg-hll_sketch_estimate)

*Summary tables store daily sketches so any period's distinct count is cheap.*

Build one HLL sketch of `user_id` per day from `events`, then return `week_start` (Monday) and `users` (the estimate from merging that week's daily sketches), ordered by `week_start`.

In [ ]:
events.select("user_id", "ts").orderBy("ts").show()

**Expected output**

| week_start | users |
| --- | --- |
| 2026-01-05 | 5 |
| 2026-01-12 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
daily = events.groupBy(F.to_date("ts").alias("day")).agg(F.hll_sketch_agg("user_id").alias("sketch"))
result = (daily.groupBy(F.to_date(F.date_trunc("week", "day")).alias("week_start"))
          .agg(F.hll_sketch_estimate(F.hll_union_agg("sketch")).alias("users")).orderBy("week_start"))
result.show()

## Exercise 13: Top category per region by revenue

**Difficulty:** Hard · about 15 min · **Practises:** [`max_by`, `min_by`](pyspark-reference-07-aggregations.ipynb#max_by-min_by)

*"Best X per Y" without a window: aggregate twice and use max_by.*

Compute line revenue per customer region (`unknown` when missing) and category, then return `region`, `top_category` and `top_revenue` for each region, ordered by `region`. Lines without a product are excluded.

In [ ]:
order_items.show(20)

**Expected output**

| region | top_category | top_revenue |
| --- | --- | --- |
| North | Garden | 260.00 |
| South | Outdoor | 310.00 |
| West | Outdoor | 210.00 |
| unknown | Pantry | 1180.00 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
by_pair = (order_items.join(products, "sku").join(orders.select("order_id", "customer_id"), "order_id")
           .join(customers.select("customer_id", "region"), "customer_id", "left")
           .groupBy(F.coalesce("region", F.lit("unknown")).alias("region"), "category")
           .agg(F.sum(F.col("qty") * F.col("unit_price")).alias("revenue")))
result = by_pair.groupBy("region").agg(F.max_by("category", "revenue").alias("top_category"), F.max("revenue").alias("top_revenue")).orderBy("region")
result.show()

## Exercise 14: Profile a table's columns

**Difficulty:** Medium · about 10 min · **Practises:** [`count`, `count_distinct`](pyspark-reference-07-aggregations.ipynb#count-count_distinct), [`approx_count_distinct`](pyspark-reference-07-aggregations.ipynb#approx_count_distinct)

*Data profiling before modelling: completeness and cardinality of every column.*

For each column of `customers`, return `column`, `non_null` and `distinct_values` (exact), in schema order.

In [ ]:
customers.show()

**Expected output**

| column | non_null | distinct_values |
| --- | --- | --- |
| customer_id | 8 | 8 |
| name | 8 | 8 |
| region | 7 | 4 |
| signup_date | 8 | 8 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
stats = customers.agg(*[F.count(c).alias(f"{c}__n") for c in customers.columns], *[F.count_distinct(c).alias(f"{c}__d") for c in customers.columns]).first()
result = spark.createDataFrame([(c, stats[f"{c}__n"], stats[f"{c}__d"]) for c in customers.columns], "column string, non_null int, distinct_values int")
result.show()

## Exercise 15: Weighted average price per category

**Difficulty:** Medium · about 10 min · **Practises:** [`sum`, `avg` / `mean`, `min`, `max`](pyspark-reference-07-aggregations.ipynb#sum-avg--mean-min-max)

*An average of prices ignores volume; the weighted average divides revenue by units.*

Per product `category`, return `simple_avg_price` (average of `unit_price` over lines) and `weighted_avg_price` (sum of qty × unit_price divided by sum of qty), both rounded to 2, ordered by `category`.

In [ ]:
order_items.join(products, "sku").select("category", "qty", "unit_price").show(20)

**Expected output**

| category | simple_avg_price | weighted_avg_price |
| --- | --- | --- |
| Garden | 110.85 | 99.78 |
| Kitchen | 41.80 | 36.93 |
| Outdoor | 173.33 | 173.33 |
| Pantry | 48.56 | 55.84 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = order_items.join(products, "sku").groupBy("category").agg(
    F.round(F.avg("unit_price"), 2).alias("simple_avg_price"),
    F.round(F.sum(F.col("qty") * F.col("unit_price")) / F.sum("qty"), 2).alias("weighted_avg_price"),
).orderBy("category")
result.show()

## Exercise 16: GROUP BY ALL with FILTER in SQL

**Difficulty:** Easy · about 5 min · **Practises:** [`GROUP BY`, `GROUP BY ALL`, `HAVING` and `FILTER`](pyspark-reference-07-aggregations.ipynb#group-by-group-by-all-having-and-filter)

*Analysts' SQL uses GROUP BY ALL and FILTER; know how to read and write it.*

In Spark SQL, return `region` (from customers, `unknown` when null), `orders` and `large_orders` (amount >= 200, using `FILTER`), grouped with `GROUP BY ALL` and ordered by `region`.

In [ ]:
print("tables: orders, customers")

**Expected output**

| region | orders | large_orders |
| --- | --- | --- |
| North | 5 | 1 |
| South | 2 | 1 |
| West | 3 | 1 |
| unknown | 2 | 1 |

In [ ]:
# Write your solution here: build a DataFrame called result.

In [ ]:
result = spark.sql("""
    SELECT coalesce(c.region, 'unknown') AS region, count(*) AS orders, count(*) FILTER (WHERE o.amount >= 200) AS large_orders
    FROM {o} o LEFT JOIN {c} c ON o.customer_id = c.customer_id
    GROUP BY ALL
    ORDER BY region
""", o=orders, c=customers)
result.show()

**Next:** look anything up in [PySpark reference 7: aggregations](pyspark-reference-07-aggregations.ipynb), or test yourself across topics with the [PySpark coding challenges](pyspark-coding-challenges.ipynb).